# Semana 10 — Construção Guiada do Fluxo Node-RED, Nó a Nó
## Da mensagem MQTT bruta ao dado industrial validado

Nas Semanas 05 e 07, observamos e importamos fluxos prontos. Nesta aula, o fluxo deixa de ser uma “caixa-preta”: cada nó será inserido, configurado, conectado e validado separadamente.

> **Questão central:** o que acontece com `msg` em cada ponto do fluxo — e como provar que a transformação está correta?

### Resultados de aprendizagem

- explicar o contrato da mensagem entre dois nós;
- configurar a conexão MQTT sem confundir broker, tópico e payload;
- observar a mensagem antes e depois da conversão JSON;
- usar `switch`, `change` e `function` com responsabilidades diferentes;
- testar cada ligação antes de ampliar o fluxo;
- reconhecer falhas de tipo, propriedade ausente e rota incorreta;
- exportar um fluxo compreensível e versionável.

![Construção do fluxo Node-RED](img/semana_10_fluxo_node_red_no_a_no.png)

## 1. O objeto `msg` é o fio condutor

No Node-RED, os fios transportam objetos JavaScript. Por convenção, o conteúdo principal está em `msg.payload`, mas outros metadados podem viajar no mesmo objeto:

```javascript
{
  topic: "smartn1/linha1/motor01/telemetria",
  payload: "{\"temperatura\": 78.4, \"vibracao\": 4.1}",
  qos: 1,
  retain: false
}
```

Antes do nó `json`, o payload pode ser texto. Depois dele, torna-se objeto. Essa diferença define se `msg.payload.temperatura` existe ou não.

In [ ]:
import json

msg = {
    "topic": "smartn1/linha1/motor01/telemetria",
    "payload": '{"maquina_id":"motor01","temperatura":78.4,"vibracao":4.1}',
    "qos": 1,
    "retain": False,
}
print("Tipo antes do JSON:", type(msg["payload"]).__name__)
msg["payload"] = json.loads(msg["payload"])
print("Tipo depois do JSON:", type(msg["payload"]).__name__)
print("Temperatura:", msg["payload"]["temperatura"])

## 2. O contrato de dados do fluxo

Um fluxo confiável define o que recebe e o que entrega. Usaremos este contrato mínimo:

| Campo | Tipo | Regra |
|---|---|---|
| `maquina_id` | texto | obrigatório e não vazio |
| `timestamp` | texto ISO 8601 | obrigatório |
| `temperatura` | número | intervalo físico de -20 a 150 °C |
| `vibracao` | número | valor não negativo |
| `estado` | texto | `operando`, `parada` ou `falha` |

Payload válido para os experimentos:

```json
{"maquina_id":"motor01","timestamp":"2026-10-06T19:00:00-03:00","temperatura":72.5,"vibracao":2.8,"estado":"operando"}
```

## 3. Nó 1 — `mqtt in`: fronteira com o mundo externo

O nó recebe eventos do broker. A conexão é mantida por um **configuration node**, compartilhável por outros nós MQTT.

Configuração conceitual:

- broker: Mosquitto da stack já utilizada;
- tópico: `smartn1/linha1/+/telemetria`;
- QoS inicial: `0`, para isolar a construção funcional antes do laboratório de confiabilidade;
- saída: string ou buffer, conforme a configuração.

O curinga `+` ocupa exatamente um nível. Assim, `motor01` e `motor02` podem chegar pela mesma assinatura, mas tópicos com níveis adicionais não serão capturados.

### Ponto de observação

Conecte inicialmente apenas um `debug` configurado para mostrar o **objeto completo**. Verifique `topic`, tipo de `payload`, `qos` e `retain`. O painel Debug é evidência do estado da mensagem, não apenas uma tela de saída.

## 4. Nó 2 — `json`: mudança explícita de representação

O nó `json` não “melhora” os dados; ele converte representação. Uma string JSON válida se transforma em objeto. Uma string inválida gera erro.

Teste os dois casos:

```text
Válido:   {"temperatura":72.5}
Inválido: {temperatura:72.5}
```

Depois do nó, um segundo `debug` deve mostrar `msg.payload` como `object`. Manter pontos de observação antes e depois reduz o espaço de busca quando algo falha.

In [ ]:
import json

amostras = ['{"temperatura":72.5}', '{temperatura:72.5}']
for payload in amostras:
    try:
        convertido = json.loads(payload)
        print("VÁLIDO  ->", convertido)
    except json.JSONDecodeError as erro:
        print("INVÁLIDO ->", erro.msg)

## 5. Nó 3 — `switch`: decisões declarativas

O `switch` deve rotear, não transformar. Configure a propriedade `msg.payload.estado` com três saídas:

1. igual a `operando`;
2. igual a `parada`;
3. caso contrário.

A terceira saída evita o desaparecimento silencioso de mensagens inesperadas. Conecte um `debug` distinto a cada saída e identifique-o pelo nome.

### Por que não usar `function` para tudo?

Um nó declarativo revela sua intenção visualmente. Código é adequado quando a regra exige cálculo ou estrutura mais elaborada. Usar o nó mais simples que expressa a regra torna o fluxo mais legível e testável.

## 6. Nó 4 — `change`: transformação sem código

Na rota de operação, use `change` para:

- mover `msg.payload.maquina_id` para `msg.maquina_id`;
- definir `msg.categoria` como `telemetria_normal`;
- preservar o payload original;
- opcionalmente criar `msg.recebido_em` com timestamp.

O `change` é preferível para atribuições, renomeações, exclusões e expressões JSONata simples. A saída deve ser inspecionada antes de seguir.

## 7. Nó 5 — `function`: validação e enriquecimento

O código abaixo pressupõe que o payload já é objeto. Ele devolve duas saídas: dados válidos e rejeições.

```javascript
const p = msg.payload;
const erros = [];

if (!p.maquina_id) erros.push("maquina_id ausente");
if (!p.timestamp) erros.push("timestamp ausente");
if (typeof p.temperatura !== "number") erros.push("temperatura não numérica");
if (typeof p.vibracao !== "number" || p.vibracao < 0) erros.push("vibração inválida");

if (erros.length > 0) {
    msg.erros = erros;
    msg.valido = false;
    return [null, msg];
}

msg.valido = true;
msg.payload.temperatura_k = Number((p.temperatura + 273.15).toFixed(2));
return [msg, null];
```

As duas saídas materializam uma decisão importante: dado inválido não deve desaparecer nem seguir como se fosse confiável.

In [ ]:
def validar(payload):
    erros = []
    if not payload.get("maquina_id"):
        erros.append("maquina_id ausente")
    if not payload.get("timestamp"):
        erros.append("timestamp ausente")
    if not isinstance(payload.get("temperatura"), (int, float)):
        erros.append("temperatura não numérica")
    if not isinstance(payload.get("vibracao"), (int, float)) or payload.get("vibracao", -1) < 0:
        erros.append("vibração inválida")
    return {"valido": not erros, "erros": erros, "payload": payload}

casos = [
    {"maquina_id":"motor01","timestamp":"2026-10-06T19:00:00-03:00","temperatura":72.5,"vibracao":2.8},
    {"maquina_id":"motor01","temperatura":"quente","vibracao":-1},
]
for caso in casos:
    print(validar(caso))

### Fluxo Node-RED Completo (JSON para Importação)

Abaixo está o código JSON completo do fluxo construído nesta aula. Você pode copiar o conteúdo abaixo e importar diretamente no Node-RED (**Menu -> Import -> Clipboard**):

```json
[
    {
        "id": "flow_semana10",
        "type": "tab",
        "label": "Semana 10 - Validação Telemetria",
        "disabled": false
    },
    {
        "id": "mqtt_in_1",
        "type": "mqtt in",
        "z": "flow_semana10",
        "name": "Entrada Telemetria",
        "topic": "smartn1/linha1/+/telemetria",
        "qos": "0",
        "datatype": "auto-detect",
        "broker": "mosquitto_broker",
        "x": 160,
        "y": 120,
        "wires": [["json_1", "debug_bruto"]]
    },
    {
        "id": "json_1",
        "type": "json",
        "z": "flow_semana10",
        "name": "String -> JSON",
        "property": "payload",
        "action": "",
        "pretty": false,
        "x": 360,
        "y": 120,
        "wires": [["switch_1"]]
    },
    {
        "id": "switch_1",
        "type": "switch",
        "z": "flow_semana10",
        "name": "Filtrar Estado",
        "property": "payload.estado",
        "propertyType": "msg",
        "rules": [
            {"t": "eq", "v": "operando", "vt": "str"},
            {"t": "eq", "v": "parada", "vt": "str"},
            {"t": "else"}
        ],
        "checkall": "true",
        "repair": false,
        "x": 540,
        "y": 120,
        "wires": [
            ["change_1"],
            ["debug_parada"],
            ["debug_desconhecido"]
        ]
    },
    {
        "id": "change_1",
        "type": "change",
        "z": "flow_semana10",
        "name": "Enriquecer Meta",
        "rules": [
            {"t": "set", "p": "maquina_id", "pt": "msg", "to": "payload.maquina_id", "tot": "msg"},
            {"t": "set", "p": "categoria", "pt": "msg", "to": "telemetria_normal", "tot": "str"}
        ],
        "action": "",
        "property": "",
        "from": "",
        "to": "",
        "reg": false,
        "x": 730,
        "y": 100,
        "wires": [["function_validar"]]
    },
    {
        "id": "function_validar",
        "type": "function",
        "z": "flow_semana10",
        "name": "Validar e Converter K",
        "func": "const p = msg.payload;\nconst erros = [];\n\nif (!p.maquina_id) erros.push(\"maquina_id ausente\");\nif (!p.timestamp) erros.push(\"timestamp ausente\");\nif (typeof p.temperatura !== \"number\") erros.push(\"temperatura não numérica\");\nif (typeof p.vibracao !== \"number\" || p.vibracao < 0) erros.push(\"vibração inválida\");\n\nif (erros.length > 0) {\n    msg.erros = erros;\n    msg.valido = false;\n    return [null, msg];\n}\n\nmsg.valido = true;\nmsg.payload.temperatura_k = Number((p.temperatura + 273.15).toFixed(2));\nreturn [msg, null];",
        "outputs": 2,
        "x": 940,
        "y": 100,
        "wires": [
            ["debug_validos"],
            ["debug_rejeitados"]
        ]
    },
    {
        "id": "debug_bruto",
        "type": "debug",
        "z": "flow_semana10",
        "name": "1. Debug Bruto",
        "active": true,
        "tosidebar": true,
        "console": false,
        "tostatus": false,
        "complete": "true",
        "targetType": "full",
        "x": 370,
        "y": 200,
        "wires": []
    },
    {
        "id": "debug_validos",
        "type": "debug",
        "z": "flow_semana10",
        "name": "Saída 1: Válidos",
        "active": true,
        "tosidebar": true,
        "console": false,
        "tostatus": false,
        "complete": "true",
        "targetType": "full",
        "x": 1160,
        "y": 80,
        "wires": []
    },
    {
        "id": "debug_rejeitados",
        "type": "debug",
        "z": "flow_semana10",
        "name": "Saída 2: Rejeitados",
        "active": true,
        "tosidebar": true,
        "console": false,
        "tostatus": false,
        "complete": "true",
        "targetType": "full",
        "x": 1170,
        "y": 140,
        "wires": []
    }
]
```


## 8. Estratégia incremental de construção

O fluxo cresce somente quando o estágio anterior produz a estrutura esperada:

| Incremento | Evidência mínima |
|---|---|
| `mqtt in → debug bruto` | tópico e payload recebidos |
| `→ json → debug objeto` | payload convertido em objeto |
| `→ switch` | cada estado chega à saída correta |
| `→ change` | propriedades movidas/definidas |
| `→ function` | válidos e rejeitados separados |
| `→ debug final` | contrato final visível |

Esse método evita montar dez nós para só então descobrir que o primeiro recebeu um payload em formato inesperado.

## 9. Casos de teste do fluxo

Publique ou injete pelo menos estes eventos:

1. operação normal;
2. máquina parada;
3. estado desconhecido;
4. temperatura como texto;
5. campo obrigatório ausente;
6. JSON malformado.

Para cada caso, registre: entrada, último nó alcançado, saída observada e interpretação. A mensagem deve ser rastreável pelo tópico e pelo identificador da máquina.

## 10. Atividade de consolidação

Construa o fluxo sem importar o JSON pronto. Ao final:

- os nomes dos nós devem expressar suas responsabilidades;
- cada transformação deve possuir um ponto de observação durante o desenvolvimento;
- mensagens inválidas devem seguir para uma saída de rejeição;
- o fluxo exportado deve ser salvo no repositório;
- uma breve descrição deve explicar o contrato entre os nós.

### Reflexão

Qual nó seria o melhor local para cada regra: `switch`, `change` ou `function`? Justifique pela legibilidade, não apenas pela possibilidade técnica.

## Bibliografia e aprofundamento

- NODE-RED. **Node-RED Concepts**. <https://nodered.org/docs/user-guide/concepts>.
- NODE-RED. **The Core Nodes**. <https://nodered.org/docs/user-guide/nodes>.
- NODE-RED. **Working with messages**. <https://nodered.org/docs/user-guide/messages>.
- NODE-RED. **Writing Functions**. <https://nodered.org/docs/user-guide/writing-functions>.
- OASIS. *MQTT Version 5.0*. 2019. <https://docs.oasis-open.org/mqtt/mqtt/v5.0/mqtt-v5.0.html>.
- BANKS, Andrew et al. *MQTT Version 5.0*. OASIS Standard, 2019.